# Cross-model comparison

Compiles the saved results of every trained model into one view for the Results
chapter. No model is trained, retrained or refitted here; this notebook only
reads result files already written to `data/` and combines them.

Result files are discovered by pattern rather than listed explicitly, so adding
a further model or condition later means dropping one more file into `data/`
without editing this notebook.

## Setup

Imports and file paths.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd().parent))
from src.evaluation_utils import SERIES_COLORS

DATA_DIR = Path("../data")
FIGURES_DIR = Path("../figures")
RESULT_GLOB = "*_results.json"
RANDOM_STATE = 42

CONDITIONS = ["no imbalance handling", "class weighting"]
METRICS = ["AUPRC", "AUROC", "recall_AKI", "specificity_non_AKI", "precision", "F1"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("result files found:", len(sorted(DATA_DIR.glob(RESULT_GLOB))))

## 1. Inspect what each result file actually contains

Every result file in `data/` is listed with its top-level keys and a check for
each block the later steps need. The structure is read from the files rather
than assumed from what the training notebooks were meant to save.

In [ ]:
def describe(path):
    """Return a dict describing what one result file contains."""
    payload = json.loads(path.read_text())
    text = json.dumps(payload).lower()
    metrics = payload.get("test_metrics", {})

    return {
        "file": path.name,
        "condition": payload.get("condition"),
        "top_level_keys": list(payload.keys()),
        "has_six_metrics": all(m in metrics for m in METRICS),
        "has_confidence_intervals": "ci_lower" in text,
        "has_calibration": any(k in text for k in ('"brier"', '"ece"', '"mce"')),
        "has_prevalence": any(k in text for k in ("prevalence", "positive_rate", "base_rate")),
        "payload": payload,
    }


files = sorted(DATA_DIR.glob(RESULT_GLOB))
described = [describe(p) for p in files]

for d in described:
    print(d["file"])
    print(f"    condition: {d['condition']!r}")
    print(f"    top-level keys: {d['top_level_keys']}")
    print(f"    six-metric summary:    {'yes' if d['has_six_metrics'] else 'NO'}")
    print(f"    confidence intervals:  {'yes' if d['has_confidence_intervals'] else 'NO'}")
    print(f"    calibration numbers:   {'yes' if d['has_calibration'] else 'NO'}")
    print(f"    prevalence recorded:   {'yes' if d['has_prevalence'] else 'NO'}")
    print()

In [ ]:
# summarise what is available across all files, so gaps are visible before use
n = len(described)
print(f"{n} result files inspected")
print(f"  with the six-metric summary:   {sum(d['has_six_metrics'] for d in described)}/{n}")
print(f"  with confidence intervals:     {sum(d['has_confidence_intervals'] for d in described)}/{n}")
print(f"  with calibration numbers:      {sum(d['has_calibration'] for d in described)}/{n}")
print(f"  with prevalence recorded:      {sum(d['has_prevalence'] for d in described)}/{n}")
print()

if not any(d["has_confidence_intervals"] for d in described):
    print("No result file stores confidence intervals. The comparison table below")
    print("reports point estimates only. Intervals are printed inside each model's")
    print("own training notebook but were not persisted to the result files.")
    print()
if not any(d["has_calibration"] for d in described):
    print("No result file stores calibration numbers (Brier, ECE, MCE), so the")
    print("calibration comparison in step 4 cannot be built from saved data.")

### Files that do not use the project's condition names

Condition values are expected to be one of the two full descriptive names.
Files using anything else are reported here and excluded from the comparison,
rather than being dropped silently or merged in under an unrecognised label.

In [ ]:
recognised_all = [d for d in described if d["condition"] in CONDITIONS]
unrecognised = [d for d in described if d["condition"] not in CONDITIONS]

# Models sharing cohort, observation window and label definition, and so
# directly comparable. temporal_ffnn predicts in a different setting, window
# level with continuous reassessment, and is reported on its own.
HEAD_TO_HEAD = ["xgboost", "static_ffnn", "gru", "static_trend"]
EXPLORATORY = ["temporal_ffnn"]


def result_model(entry):
    """Model name for one result-file entry."""
    stem = entry["file"].replace("_results.json", "")
    suffix = "_" + entry["condition"].replace(" ", "_")
    return stem[: -len(suffix)] if stem.endswith(suffix) else stem


def head_to_head(frame):
    """Restrict a table to the directly comparable models."""
    return frame[frame["model"].isin(HEAD_TO_HEAD)].reset_index(drop=True)


# every section below this point works from the head-to-head list; the lift
# table and the exploratory section use recognised_all explicitly
recognised = [d for d in recognised_all if result_model(d) in HEAD_TO_HEAD]
exploratory_recognised = [d for d in recognised_all if result_model(d) in EXPLORATORY]

if unrecognised:
    print("excluded from the comparison:")
    for d in unrecognised:
        print(f"  {d['file']}  condition={d['condition']!r}")
    print()
    print("These use an older naming scheme. Their metrics may duplicate a")
    print("recognised file; check before deleting.")
else:
    print("every result file uses a recognised condition name")

print()
print(f"result files recognised: {len(recognised_all)}")
print(f"  head-to-head comparison: {len(recognised)} "
      f"({', '.join(sorted({result_model(d) for d in recognised}))})")
print(f"  reported separately:     {len(exploratory_recognised)} "
      f"({', '.join(sorted({result_model(d) for d in exploratory_recognised}))})")

## 2. Compile the six-metric comparison table

One row per model and condition, read from whatever files are present. The
model name comes from the file name, with the condition suffix removed.

In [ ]:
def model_name(filename, condition):
    """Derive the model name from a result file name."""
    stem = filename.replace("_results.json", "")
    suffix = "_" + condition.replace(" ", "_")
    return stem[: -len(suffix)] if stem.endswith(suffix) else stem


rows = []
for d in recognised_all:
    metrics = d["payload"].get("test_metrics", {})
    row = {
        "model": model_name(d["file"], d["condition"]),
        "condition": d["condition"],
    }
    row.update({m: metrics.get(m, np.nan) for m in METRICS})
    rows.append(row)

comparison_all = pd.DataFrame(rows).sort_values(["model", "condition"]).reset_index(drop=True)
comparison = head_to_head(comparison_all)

print(f"{len(comparison_all)} model/condition combinations in total")
print(f"{len(comparison)} in the head-to-head comparison: "
      f"{', '.join(sorted(set(comparison['model'])))}")
held_out = sorted(set(comparison_all["model"]) - set(HEAD_TO_HEAD))
if held_out:
    print(f"reported separately: {', '.join(held_out)}")
comparison.round(4)

In [ ]:
# which model/condition combinations are missing
present = set(zip(comparison["model"], comparison["condition"]))
expected = {(m, c) for m in comparison["model"].unique() for c in CONDITIONS}
missing = sorted(expected - present)

if missing:
    print("missing model/condition combinations:")
    for m, c in missing:
        print(f"  {m} / {c}")
else:
    print("every model present has results for both conditions")

## 3. Prevalence-normalized comparison

AUPRC depends on the positive rate of the data it is computed on, so it is not
directly comparable across representations with different prevalence. The
stay-level representations label roughly 60% of rows positive; the
sliding-window representation labels a much smaller share positive.

Lift is `AUPRC / prevalence`, where prevalence is the model's own positive rate
in its own test set. An uninformative model scores an AUPRC equal to the
prevalence, so a lift of 1.0 is the no-skill baseline regardless of the
underlying balance (Flach, P. & Kull, M. (2015). Precision-Recall-Gain Curves:
PR Analysis Done Right. Advances in Neural Information Processing Systems 28,
838-846).

No result file records its prevalence, so each is recomputed here by rebuilding
that model's test split from the representation file, using the same split
parameters as the training notebook.

In [ ]:
def stay_level_prevalence(parquet_name):
    """Test positive rate for a stay-level 80/20 stratified split."""
    df = pd.read_parquet(DATA_DIR / parquet_name)
    _, test_df = train_test_split(
        df, test_size=0.20, stratify=df["label"], random_state=RANDOM_STATE
    )
    return float(test_df["label"].mean()), len(test_df)


def window_level_prevalence(parquet_name):
    """Test positive rate for a patient-grouped 80/20 split over windows."""
    df = pd.read_parquet(DATA_DIR / parquet_name)
    patient_labels = df.groupby("subject_id")["label"].max()
    _, test_patients = train_test_split(
        patient_labels.index.to_numpy(), test_size=0.20,
        stratify=patient_labels.to_numpy(), random_state=RANDOM_STATE,
    )
    test_df = df[df["subject_id"].isin(set(test_patients))]
    return float(test_df["label"].mean()), len(test_df)


# how each model's test set is rebuilt; add an entry when a new model is added
PREVALENCE_SOURCES = {
    "xgboost": lambda: stay_level_prevalence("static_representation.parquet"),
    "static_ffnn": lambda: stay_level_prevalence("static_representation.parquet"),
    "static_trend": lambda: stay_level_prevalence("static_trend_representation.parquet"),
    "gru": lambda: stay_level_prevalence("sequence_static.parquet"),
    "temporal_ffnn": lambda: window_level_prevalence("sliding_window_representation.parquet"),
}

prevalence = {}
for model in comparison_all["model"].unique():
    source = PREVALENCE_SOURCES.get(model)
    if source is None:
        print(f"{model}: no prevalence source registered, lift cannot be computed")
        prevalence[model] = (np.nan, 0)
        continue
    rate, n_rows = source()
    prevalence[model] = (rate, n_rows)
    print(f"{model:16s} test rows {n_rows:6,d}   positive rate {rate * 100:6.2f}%")

> Note: temporal_ffnn appears in the lift table below but not in the
> head-to-head comparisons. It predicts in a different setting, reassessing a
> patient repeatedly across their stay rather than making one prediction from a
> single 24-hour window, and its rows are windows rather than patients. Lift
> divides AUPRC by each model's own positive rate, which is what makes a figure
> comparable across those different settings, so its inclusion here is for
> context on relative predictive skill only, not a head-to-head ranking.

In [ ]:
comparison_all["prevalence"] = comparison_all["model"].map(lambda m: prevalence[m][0])
comparison_all["test_rows"] = comparison_all["model"].map(lambda m: prevalence[m][1])
comparison_all["lift"] = comparison_all["AUPRC"] / comparison_all["prevalence"]

ordered = ["model", "condition"] + METRICS + ["prevalence", "lift", "test_rows"]
comparison_all = comparison_all[ordered]
comparison = head_to_head(comparison_all)

# every model, temporal_ffnn included, since lift is what makes the
# comparison across different positive rates meaningful
comparison_all[["model", "condition", "AUPRC", "prevalence", "lift"]].round(4)

### Confidence intervals

The interval for each metric is read from the result files, which store the
bootstrap bounds computed in each training notebook. A display column per metric
combines the point estimate with its interval, matching the format each training
notebook prints.

In [ ]:
def interval_lookup(described_entry, metric):
    """Return (ci_lower, ci_upper) for one metric, or (nan, nan) if absent."""
    block = described_entry["payload"].get("confidence_intervals", {})
    entry = block.get(metric)
    if not entry:
        return np.nan, np.nan
    return float(entry["ci_lower"]), float(entry["ci_upper"])


by_key = {(model_name(d["file"], d["condition"]), d["condition"]): d for d in recognised_all}

for metric in METRICS:
    lows, highs = [], []
    for _, row in comparison_all.iterrows():
        low, high = interval_lookup(by_key[(row["model"], row["condition"])], metric)
        lows.append(low)
        highs.append(high)
    comparison_all[f"{metric}_ci_lower"] = lows
    comparison_all[f"{metric}_ci_upper"] = highs

comparison = head_to_head(comparison_all)

missing_ci = [m for m in METRICS if comparison_all[f"{m}_ci_lower"].isna().any()]
if missing_ci:
    print("confidence intervals missing for:", missing_ci)
else:
    print(f"confidence intervals present for all {len(METRICS)} metrics, "
          f"all {len(comparison_all)} model/condition combinations")

In [ ]:
def with_intervals(frame):
    """Combine each metric with its interval into one display string."""
    out = frame[["model", "condition"]].copy()
    for metric in METRICS:
        out[metric] = [
            f"{p:.4f} [{lo:.4f}, {hi:.4f}]" if pd.notna(lo) else f"{p:.4f}"
            for p, lo, hi in zip(frame[metric], frame[f"{metric}_ci_lower"],
                                 frame[f"{metric}_ci_upper"])
        ]
    out["prevalence"] = frame["prevalence"].round(4)
    out["lift"] = frame["lift"].round(4)
    return out


comparison_display = with_intervals(comparison)
comparison_display

## 4. Calibration comparison

Built only if calibration numbers are present in the result files. Where they
are absent, the models concerned are named rather than omitted, so the gap is
visible.

In [ ]:
calibration_rows = []
missing_calibration = []

for d in recognised:
    payload = d["payload"]
    block = payload.get("calibration") or {}
    values = {k: block.get(k) for k in ("brier", "ece", "mce")}
    if d["has_calibration"] and any(v is not None for v in values.values()):
        calibration_rows.append({
            "model": model_name(d["file"], d["condition"]),
            "condition": d["condition"],
            **{k.upper(): v for k, v in values.items()},
        })
    else:
        missing_calibration.append(f"{model_name(d['file'], d['condition'])} / {d['condition']}")

if calibration_rows:
    calibration_table = pd.DataFrame(calibration_rows).sort_values(["model", "condition"])
else:
    calibration_table = None

if missing_calibration:
    print("calibration numbers not saved for:")
    for entry in missing_calibration:
        print(f"  {entry}")
    print()
    print("Brier, ECE and MCE are printed by calibration_report inside each")
    print("training notebook but are not written to the result files, so the")
    print("calibration comparison cannot be compiled here. Adding them to")
    print("save_results in each training notebook would make this table possible.")
else:
    print(calibration_table.round(4).to_string(index=False))

print()
print(f"calibration table rows: {0 if calibration_table is None else len(calibration_table)}"
      f"  models: {sorted(calibration_table['model'].unique()) if calibration_table is not None else []}")


### Completeness check

Every model and condition must have both calibration numbers and confidence
intervals before the comparison table is treated as final. Anything missing is
named here rather than passed over.

In [ ]:
incomplete = []
for d in recognised_all:
    key = f"{model_name(d['file'], d['condition'])} / {d['condition']}"
    if not d["has_calibration"]:
        incomplete.append(f"{key}: calibration missing")
    if not d["has_confidence_intervals"]:
        incomplete.append(f"{key}: confidence intervals missing")

if incomplete:
    print("INCOMPLETE, the comparison below rests on partial data:")
    for entry in incomplete:
        print(f"  {entry}")
else:
    print(f"complete: all {len(recognised_all)} model/condition combinations have "
          f"calibration numbers and confidence intervals")

## 5. Summary chart

One group per model, one bar per condition, showing lift. The dashed line marks
a lift of 1.0, the score an uninformative model would reach on its own data.

In [ ]:
chart_data = comparison.pivot(index="model", columns="condition", values="lift")
chart_data = chart_data.reindex(columns=[c for c in CONDITIONS if c in chart_data.columns])

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(chart_data.index))
width = 0.8 / max(len(chart_data.columns), 1)

for i, condition in enumerate(chart_data.columns):
    offset = (i - (len(chart_data.columns) - 1) / 2) * width
    ax.bar(x + offset, chart_data[condition], width, label=condition,
           color=SERIES_COLORS[i])

ax.axhline(1.0, linestyle="--", color="gray", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(chart_data.index)
ax.set_ylabel("lift (AUPRC / prevalence)")
ax.set_title("Model comparison: AUPRC relative to each model's own prevalence")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))

plt.show()

## Additional graphs

Three further views of the same table: every metric side by side, the effect of
class weighting on specificity, and raw AUPRC against prevalence-normalized
lift.

In [ ]:
fig_grid, axes = plt.subplots(2, 3, figsize=(17.5, 9.3))

# Sizes are set per element rather than by one multiplier. Tick labels were the
# illegible part at column width, so they get the largest increase; subplot
# titles a modest one; the figure title and legend were already legible and
# keep their original size.
TICK_SIZE = 13        # was 8 on x, matplotlib default on y
SUBPLOT_TITLE = 15    # was the matplotlib default of 12
LEGEND_SIZE = 11      # raised toward the tick size, still below it
SUPTITLE_SIZE = 17    # figure title, a step above the subplot titles

for ax, metric in zip(axes.ravel(), METRICS):
    pivot = comparison.pivot(index="model", columns="condition", values=metric)
    pivot = pivot.reindex(columns=[c for c in CONDITIONS if c in pivot.columns])
    x = np.arange(len(pivot.index))
    width = 0.8 / max(len(pivot.columns), 1)
    for i, condition in enumerate(pivot.columns):
        offset = (i - (len(pivot.columns) - 1) / 2) * width
        ax.bar(x + offset, pivot[condition], width, label=condition,
               color=SERIES_COLORS[i])
    ax.set_xticks(x)
    # steeper angle and right-anchored, so the longer model names stay separated
    # at the larger tick size instead of running into each other
    ax.set_xticklabels(pivot.index, rotation=40, ha="right",
                       rotation_mode="anchor", fontsize=TICK_SIZE)
    ax.tick_params(axis="x", pad=2)
    ax.tick_params(axis="y", labelsize=TICK_SIZE)
    ax.set_title(metric, fontsize=SUBPLOT_TITLE)
    ax.set_ylim(0, 1)

# one figure-level legend in a narrow strip between the title and the panels,
# so it belongs to the whole grid rather than sitting inside one panel or in
# the inter-panel gap
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig_grid.legend(handles, labels, loc="upper right", ncol=1,
                fontsize=LEGEND_SIZE, frameon=True, framealpha=1.0,
                edgecolor="0.6", borderpad=0.6,
                bbox_to_anchor=(0.995, 1.005))
fig_grid.suptitle("Every metric, by model and condition", y=0.985,
                  fontsize=SUPTITLE_SIZE)
fig_grid.tight_layout()
# tight_layout reserves generous padding for the rotated tick labels, which
# pushes the panels apart; pull them back together and reserve just enough
# headroom for the title and legend strip
fig_grid.subplots_adjust(wspace=0.25, hspace=0.55, top=0.87)
plt.show()

In [ ]:
spec = comparison.pivot(index="model", columns="condition",
                        values="specificity_non_AKI")
spec = spec.reindex(columns=[c for c in CONDITIONS if c in spec.columns])

fig_spec, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(spec.index))
width = 0.8 / max(len(spec.columns), 1)
for i, condition in enumerate(spec.columns):
    offset = (i - (len(spec.columns) - 1) / 2) * width
    bars = ax.bar(x + offset, spec[condition], width, label=condition,
                  color=SERIES_COLORS[i])
    ax.bar_label(bars, fmt="%.3f", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(spec.index)
ax.set_ylabel("specificity (non-AKI)")
ax.set_title("Specificity before and after class weighting")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
fig_pair, (ax_raw, ax_lift) = plt.subplots(1, 2, figsize=(15, 5.8))

# same per-element sizing as the metric grid: tick labels carry the largest
# increase, axis and subplot titles a modest one, legend just below the ticks
TICK_SIZE = 13
AXIS_LABEL_SIZE = 14
SUBPLOT_TITLE = 15
SUPTITLE_SIZE = 17
LEGEND_SIZE = 11

for ax, column, title in [
    (ax_raw, "AUPRC", "Raw AUPRC"),
    (ax_lift, "lift", "Lift (AUPRC / prevalence)"),
]:
    pivot = comparison.pivot(index="model", columns="condition", values=column)
    pivot = pivot.reindex(columns=[c for c in CONDITIONS if c in pivot.columns])
    x = np.arange(len(pivot.index))
    width = 0.8 / max(len(pivot.columns), 1)
    for i, condition in enumerate(pivot.columns):
        offset = (i - (len(pivot.columns) - 1) / 2) * width
        ax.bar(x + offset, pivot[condition], width, label=condition,
               color=SERIES_COLORS[i])
    ax.set_xticks(x)
    # steeper angle so the longer model names stay separated at the larger size
    ax.set_xticklabels(pivot.index, rotation=40, ha="right",
                       rotation_mode="anchor", fontsize=TICK_SIZE)
    ax.tick_params(axis="x", pad=2)
    ax.tick_params(axis="y", labelsize=TICK_SIZE)
    ax.set_title(title, fontsize=SUBPLOT_TITLE)

ax_lift.axhline(1.0, linestyle="--", color="gray", linewidth=1)
ax_raw.set_ylabel("AUPRC", fontsize=AXIS_LABEL_SIZE)
ax_lift.set_ylabel("lift", fontsize=AXIS_LABEL_SIZE)

# one boxed figure-level legend in the top right, beside the title, rather than
# inside a panel or in the gap between them
handles, labels = ax_raw.get_legend_handles_labels()
fig_pair.legend(handles, labels, loc="upper right", ncol=1,
                fontsize=LEGEND_SIZE, frameon=True, framealpha=1.0,
                edgecolor="0.6", borderpad=0.6,
                bbox_to_anchor=(0.995, 1.10))
# y matches the legend box's vertical centre so the two sit level
fig_pair.suptitle("Raw AUPRC compared with prevalence-normalized lift",
                  y=1.05, va="center", fontsize=SUPTITLE_SIZE)
fig_pair.tight_layout()
# pull the two panels together and reserve headroom for the title strip
fig_pair.subplots_adjust(wspace=0.22, top=0.84)
plt.show()

## Feature importance comparison

Each model's importance file is loaded and its ten highest-ranked features under
no imbalance handling are shown side by side. XGBoost reports native gain;
the three networks report permutation importance, so values are comparable in
rank within a model but not in magnitude across models.

In [ ]:
IMPORTANCE_GLOB = "*_feature_importance.json"
IMPORTANCE_CONDITION = "no imbalance handling"
TOP_N = 10

importance_files = sorted(DATA_DIR.glob(IMPORTANCE_GLOB))
print(f"importance files found: {len(importance_files)}")

importance_by_model = {}
for path in importance_files:
    payload = json.loads(path.read_text())
    table = payload["importance"].get(IMPORTANCE_CONDITION)
    if table is None:
        print(f"  {path.name}: no '{IMPORTANCE_CONDITION}' entry, skipped")
        continue
    frame = (pd.Series(table, name="importance")
               .rename_axis("feature").reset_index()
               .sort_values("importance", ascending=False)
               .head(TOP_N).reset_index(drop=True))
    importance_by_model[payload["model"]] = (payload["method"], frame)
    print(f"  {payload['model']:16s} method={payload['method']:11s} "
          f"features={len(payload['feature_names'])}")

exploratory_importance = {m: v for m, v in importance_by_model.items()
                          if m in EXPLORATORY}
importance_by_model = {m: v for m, v in importance_by_model.items()
                       if m in HEAD_TO_HEAD}

if exploratory_importance:
    print()
    print(f"held back for the exploratory section: "
          f"{', '.join(sorted(exploratory_importance))}")

missing_importance = sorted(set(comparison["model"]) - set(importance_by_model))
if missing_importance:
    print()
    print("no importance file for:", missing_importance)

In [ ]:
combined = pd.DataFrame({"rank": range(1, TOP_N + 1)})
for model, (method, frame) in sorted(importance_by_model.items()):
    combined[f"{model} ({method})"] = [
        f"{row.feature}  {row.importance:.4f}" for row in frame.itertuples()
    ]

combined.set_index("rank")

print(f"feature importance columns: {[c for c in combined.columns if c != 'rank']}")


In [ ]:
# how much of each model's top ten is trend-based, the temporal representation
# being the only one whose features encode change over time
print("share of the top ten that are trend features")
for model, (_, frame) in sorted(importance_by_model.items()):
    trend = frame["feature"].str.endswith("_trend").sum()
    print(f"  {model:16s} {trend}/{TOP_N}")

### Figure: feature importance across models

One panel per model, each showing its own ten highest-ranked features on its own
axis, no-imbalance-handling condition.

Separate panels rather than one shared chart because the models do not share a
feature vocabulary. The GRU consumes one column per variable across the whole
24-hour sequence, so its features are raw names such as `urine_output`, while
the other four use engineered columns such as `urine_output_last`. Only
`gender_male` and `admission_type_elective` are common to all five, so a single
shared category axis would leave four of the five models with nothing to draw.
The overlap count printed below the figure is taken across the five top-ten
lists: no single feature appears in all of them.

XGBoost reports native gain and the others report permutation importance, so
magnitudes are not comparable across panels; the ranking within each panel is.

In [ ]:
from src.evaluation_utils import plot_feature_importance_grid

# every model, so the exploratory one can be seen alongside the four
all_importance = dict(importance_by_model)
all_importance.update(exploratory_importance)

if all_importance:
    fig_importance = plot_feature_importance_grid(
        [(frame["feature"], frame["importance"], f"{model} ({method})")
         for model, (method, frame) in sorted(all_importance.items())],
        top_n=10,
        title=f"Feature importance across models ({IMPORTANCE_CONDITION})",
    )
    FIGURES_DIR.mkdir(exist_ok=True)
    path = FIGURES_DIR / "model_comparison_feature_importance.png"
    fig_importance.savefig(path, dpi=200, bbox_inches="tight")
    print(f"wrote {path.name}")

    # the shared-axis version is kept for within-model use, where both
    # conditions do share a feature set
    overlap = set.intersection(*(set(frame["feature"]) for _, frame in all_importance.values()))
    print(f"features common to all {len(all_importance)} models: {len(overlap)}")
else:
    print("no importance files loaded")

## Lead-time comparison

Lead time is how far ahead of AKI onset a correctly identified case is flagged,
measured only on true positives. It is saved by the three neural networks trained
on stay-level representations (GRU, static FFNN and static FFNN with trend
features), where one row is one patient and onset time is defined for that
row. The XGBoost notebook does not compute it, and it is not defined for the
window-level temporal-feature FFNN.

The research question asks whether the sequence model identifies cases earlier.
Non-overlapping intervals with the GRU higher would be evidence for that.
Overlapping intervals mean no clear timing difference, which is a reportable
result in itself.

In [ ]:
LEAD_KEYS = ["lead_time_median", "lead_time_ci_lower", "lead_time_ci_upper", "lead_time_n"]

lead_rows = []
without_lead_time = []
for d in recognised:
    model = model_name(d["file"], d["condition"])
    payload = d["payload"]
    if not all(k in payload for k in LEAD_KEYS):
        without_lead_time.append(f"{model} / {d['condition']}")
        continue
    lead_rows.append({
        "model": model,
        "condition": d["condition"],
        "median_hours": payload["lead_time_median"],
        "ci_lower": payload["lead_time_ci_lower"],
        "ci_upper": payload["lead_time_ci_upper"],
        "n": payload["lead_time_n"],
    })

lead_time_table = pd.DataFrame(lead_rows).sort_values(["model", "condition"]) if lead_rows else None

if without_lead_time:
    print("no lead time saved for:")
    for entry in without_lead_time:
        print(f"  {entry}")
    print()

if lead_time_table is not None:
    print(lead_time_table.round(2).to_string(index=False))
else:
    print("no model has lead-time results saved")

In [ ]:
def overlaps(first, second):
    """True if two intervals share any range."""
    return first["ci_lower"] <= second["ci_upper"] and second["ci_lower"] <= first["ci_upper"]


PAIR = ("gru", "static_ffnn")

if lead_time_table is not None:
    for condition in CONDITIONS:
        rows = lead_time_table[lead_time_table["condition"] == condition]
        available = {r["model"]: r for _, r in rows.iterrows()}
        if not all(m in available for m in PAIR):
            print(f"{condition}: both of {PAIR} needed, have {sorted(available)}")
            continue

        gru, static = available[PAIR[0]], available[PAIR[1]]
        print(f"{condition}")
        for label, row in [(PAIR[0], gru), (PAIR[1], static)]:
            print(f"  {label:12s} {row['median_hours']:.1f} h "
                  f"[{row['ci_lower']:.1f}, {row['ci_upper']:.1f}]  n = {int(row['n']):,}")

        difference = gru["median_hours"] - static["median_hours"]
        if overlaps(gru, static):
            print(f"  intervals overlap: no clear timing difference "
                  f"(GRU {difference:+.1f} h)")
        elif difference > 0:
            print(f"  intervals do not overlap, GRU higher by {difference:.1f} h: "
                  f"evidence the sequence model identifies cases earlier")
        else:
            print(f"  intervals do not overlap, GRU lower by {abs(difference):.1f} h: "
                  f"evidence against the sequence model identifying cases earlier")
        print()

## RQ1: does a trend representation improve on aggregates

The static FFNN and the static FFNN with trend features share cohort,
observation window, label definition and row structure. They differ only in how
each feature is described: min, max, mean and last against first, last and
trend. Any difference between them is attributable to that.

AUPRC and recall are the metrics the question asks about. A difference supported
by non-overlapping confidence intervals is evidence; overlapping intervals mean
no clear difference, which is a reportable outcome in itself.

In [ ]:
RQ1_PAIR = {"aggregate features": "static_ffnn", "trend features": "static_trend"}
RQ1_METRICS = ["AUPRC", "recall_AKI"]


def metric_interval(row, metric):
    """Return (point, ci_lower, ci_upper) for one metric of one row."""
    return row[metric], row[f"{metric}_ci_lower"], row[f"{metric}_ci_upper"]


for condition in CONDITIONS:
    rows = comparison[comparison["condition"] == condition].set_index("model")
    absent = [m for m in RQ1_PAIR.values() if m not in rows.index]
    if absent:
        print(f"{condition}: not comparable, missing {absent}")
        print()
        continue

    print(condition)
    for metric in RQ1_METRICS:
        agg_p, agg_lo, agg_hi = metric_interval(rows.loc[RQ1_PAIR["aggregate features"]], metric)
        trend_p, trend_lo, trend_hi = metric_interval(rows.loc[RQ1_PAIR["trend features"]], metric)
        difference = trend_p - agg_p

        print(f"  {metric}")
        print(f"    aggregate features  {agg_p:.4f} [{agg_lo:.4f}, {agg_hi:.4f}]")
        print(f"    trend features      {trend_p:.4f} [{trend_lo:.4f}, {trend_hi:.4f}]")

        if pd.isna(agg_lo) or pd.isna(trend_lo):
            print(f"    difference {difference:+.4f}, no intervals saved to judge it")
        elif agg_lo <= trend_hi and trend_lo <= agg_hi:
            print(f"    difference {difference:+.4f}, intervals overlap: "
                  f"no clear difference")
        else:
            direction = "trend features higher" if difference > 0 else "aggregate features higher"
            print(f"    difference {difference:+.4f}, intervals do not overlap: "
                  f"{direction}")
    print()

## Subgroup performance: specificity by age band

Specificity across the three age bands, one line per model, read from the
subgroup results saved by each training notebook. Both conditions are shown as
separate panels. temporal_ffnn is excluded here and appears in the exploratory
section instead.

Where a result file has no saved subgroup block, the model is named rather than
dropped silently. Subgroup results are written by `save_results`, so a file
produced before that was added will not have them until its notebook is re-run.

In [ ]:
from src.evaluation_utils import plot_subgroup_specificity

AGE_BANDS = ["18-40", "40-65", "65-90"]
SUBGROUP_ATTRIBUTE = "Age band"
SUBGROUP_METRIC = "specificity_non_AKI"


def subgroup_series(entries, attribute, metric, order):
    """Pull one metric across ordered subgroups, per model, from saved results."""
    series, absent = {}, []
    for d in entries:
        block = d["payload"].get("subgroups", {}).get(attribute)
        if not block:
            absent.append(f"{result_model(d)} / {d['condition']}")
            continue
        values = [block.get(band, {}).get(metric) for band in order]
        if any(v is None for v in values):
            absent.append(f"{result_model(d)} / {d['condition']} (incomplete)")
            continue
        series.setdefault(d["condition"], []).append((order, values, result_model(d)))
    return series, absent


head_to_head_series, missing_subgroups = subgroup_series(
    recognised, SUBGROUP_ATTRIBUTE, SUBGROUP_METRIC, AGE_BANDS)

if missing_subgroups:
    print("no saved subgroup results for:")
    for entry in missing_subgroups:
        print(f"  {entry}")
    print()
    print("re-run those training notebooks so save_results persists them")
    print()

for condition in CONDITIONS:
    plotted = head_to_head_series.get(condition)
    if not plotted:
        print(f"{condition}: nothing to plot")
        continue
    fig_subgroup = plot_subgroup_specificity(
        plotted, metric=SUBGROUP_METRIC,
        title=f"Specificity by age band, {condition}",
    )
    FIGURES_DIR.mkdir(exist_ok=True)
    path = FIGURES_DIR / f"model_comparison_specificity_age_{condition.replace(' ', '_')}.png"
    fig_subgroup.savefig(path, dpi=200, bbox_inches="tight")
    print(f"wrote {path.name}")

## Decision curve comparison

Net benefit across thresholds for the four directly comparable models, read
from the values saved in each result file. The no-imbalance-handling condition
only, so four series stay readable on one chart. A model is worth using over the
range where its curve sits above both reference lines.

In [ ]:
DCA_CONDITION = "no imbalance handling"

dca_entries = [d for d in recognised if d["condition"] == DCA_CONDITION
               and "decision_curve" in d["payload"]]
dca_missing = [result_model(d) for d in recognised if d["condition"] == DCA_CONDITION
               and "decision_curve" not in d["payload"]]

if dca_missing:
    print("no decision curve saved for:", dca_missing)

if not dca_entries:
    print("no decision curve data available for the head-to-head models")
else:
    fig_dca_models, ax = plt.subplots(figsize=(8, 5.5))
    reference = dca_entries[0]["payload"]["decision_curve"]
    ax.plot(reference["threshold"], reference["net_benefit_all"],
            linestyle="--", color="gray", label="Treat everyone")
    ax.axhline(0.0, linestyle=":", color="black", label="Treat no one")

    for i, d in enumerate(sorted(dca_entries, key=result_model)):
        saved = d["payload"]["decision_curve"]
        ax.plot(saved["threshold"], saved["net_benefit_model"],
                color=SERIES_COLORS[i], label=result_model(d))

    highest = max(max(d["payload"]["decision_curve"]["net_benefit_model"])
                  for d in dca_entries)
    ax.set_ylim(-0.05, max(highest, max(reference["net_benefit_all"])) * 1.15)
    ax.set_xlim(0, 1)
    ax.set_xlabel("threshold probability")
    ax.set_ylabel("net benefit")
    ax.set_title(f"Decision curve, head-to-head models ({DCA_CONDITION})")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))

    FIGURES_DIR.mkdir(exist_ok=True)
    dca_models_path = FIGURES_DIR / "model_comparison_decision_curve.png"
    fig_dca_models.savefig(dca_models_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"wrote {dca_models_path.name}")
    print()

    for d in sorted(dca_entries, key=result_model):
        saved = d["payload"]["decision_curve"]
        frame = pd.DataFrame(saved)
        ahead = frame[(frame["net_benefit_model"] > frame["net_benefit_all"])
                      & (frame["net_benefit_model"] > 0)]["threshold"]
        if len(ahead):
            print(f"  {result_model(d):16s} ahead of both references "
                  f"between {ahead.min():.2f} and {ahead.max():.2f}")
        else:
            print(f"  {result_model(d):16s} never ahead of both references")

### Scope check

Confirms which models reach each shared output. temporal_ffnn must appear only
in the lift table and in the exploratory section below.

In [ ]:
# SCOPE CHECK: what actually reaches each shared output
shared_outputs = {
    "Step 2 six-metric table": sorted(comparison["model"].unique()),
    "Step 2 display with intervals": sorted(comparison_display["model"].unique()),
    "Step 4 calibration table": ([] if calibration_table is None
                                 else sorted(calibration_table["model"].unique())),
    "Step 5 summary chart": sorted(chart_data.index),
    "graph: metric grid": sorted(comparison["model"].unique()),
    "graph: specificity": sorted(spec.index),
    "graph: AUPRC vs lift": sorted(comparison["model"].unique()),
    "feature importance combined": sorted(importance_by_model),
}

print(f"{'shared output':32s} {'rows':>5s}  models")
leaked = []
for label, models in shared_outputs.items():
    rows = len(comparison) if "chart" not in label and "graph" not in label else len(models)
    if label == "Step 4 calibration table":
        rows = 0 if calibration_table is None else len(calibration_table)
    elif label == "feature importance combined":
        rows = len(models)
    elif "graph" in label or "chart" in label:
        rows = len(models)
    print(f"  {label:30s} {rows:>5d}  {', '.join(models)}")
    if any(m in EXPLORATORY for m in models):
        leaked.append(label)

print()
print(f"{'lift table (temporal expected)':32s} {len(comparison_all):>5d}  "
      f"{', '.join(sorted(comparison_all['model'].unique()))}")

print()
if leaked:
    print("PROBLEM, exploratory model present in:", leaked)
else:
    print(f"no shared output includes {', '.join(EXPLORATORY)}")

# Temporal-feature FFNN (exploratory)

Reported separately from the four models above, and not benchmarked against
them. It answers a different question: rather than making one prediction from a
single 24-hour window per patient, it reassesses a patient repeatedly across
their stay, scoring each non-overlapping 24-hour window in turn. Its rows are
windows, not patients, its positive rate is much lower, and its labels are
bounded to the last clean window before AKI.

Placing it in the same table or chart as the other four would invite a direct
performance reading that the difference in setting does not support. Its results
are given here on their own terms.

In [ ]:
exploratory_entries = exploratory_recognised

if not exploratory_entries:
    print("no exploratory model results found")
else:
    for name in sorted({model_name(d["file"], d["condition"]) for d in exploratory_entries}):
        rows_all = comparison_all[comparison_all["model"] == name]
        print(f"{name}: {len(rows_all)} conditions, "
              f"{int(rows_all['test_rows'].iloc[0]):,} test windows, "
              f"positive rate {rows_all['prevalence'].iloc[0] * 100:.2f}%")

### Six metrics with confidence intervals

In [ ]:
exploratory_metrics = comparison_all[comparison_all["model"].isin(EXPLORATORY)]
exploratory_display = with_intervals(exploratory_metrics)
exploratory_display

### Calibration

In [ ]:
exploratory_calibration = pd.DataFrame([
    {
        "model": model_name(d["file"], d["condition"]),
        "condition": d["condition"],
        **{k.upper(): v for k, v in (d["payload"].get("calibration") or {}).items()},
    }
    for d in exploratory_entries
]).sort_values(["model", "condition"])

exploratory_calibration.round(4)

### Decision curve

Net benefit across the full range of thresholds, read from the values saved in
the result file. The two reference strategies are flagging every window and
flagging none. This model is ahead over the range where its curve sits above
both reference lines.

In [ ]:
def curve_frame(entry):
    """Rebuild a decision curve from the values stored in a result file."""
    saved = entry["payload"]["decision_curve"]
    return pd.DataFrame({
        "threshold": saved["threshold"],
        "net_benefit_model": saved["net_benefit_model"],
        "net_benefit_all": saved["net_benefit_all"],
    })


with_curve = [d for d in exploratory_entries if "decision_curve" in d["payload"]]
without_curve = [d["condition"] for d in exploratory_entries
                 if "decision_curve" not in d["payload"]]

if without_curve:
    print("no decision curve saved for:", without_curve)

if not with_curve:
    print("no decision curve data available for this model")
else:
    curves = {d["condition"]: curve_frame(d) for d in with_curve}
    reference = next(iter(curves.values()))

    fig_exploratory_dca, ax = plt.subplots(figsize=(7, 5))
    ax.plot(reference["threshold"], reference["net_benefit_all"],
            linestyle="--", color="gray", label="Treat everyone")
    ax.axhline(0.0, linestyle=":", color="black", label="Treat no one")
    for i, (condition, frame) in enumerate(curves.items()):
        ax.plot(frame["threshold"], frame["net_benefit_model"], label=condition,
                color=SERIES_COLORS[i])

    highest = max(frame["net_benefit_model"].max() for frame in curves.values())
    ax.set_ylim(-0.05, max(highest, reference["net_benefit_all"].max()) * 1.15)
    ax.set_xlim(0, 1)
    ax.set_xlabel("threshold probability")
    ax.set_ylabel("net benefit")
    ax.set_title("Decision curve, temporal-feature FFNN")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))

    FIGURES_DIR.mkdir(exist_ok=True)
    dca_path = FIGURES_DIR / "temporal_ffnn_decision_curve.png"
    fig_exploratory_dca.savefig(dca_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"wrote {dca_path.name}")
    print()

    for condition, frame in curves.items():
        band = frame[(frame["threshold"] >= 0.4) & (frame["threshold"] <= 0.7)]
        ahead = frame[(frame["net_benefit_model"] > frame["net_benefit_all"])
                      & (frame["net_benefit_model"] > 0)]["threshold"]
        print(condition)
        if len(ahead):
            print(f"  ahead of both references between "
                  f"{ahead.min():.2f} and {ahead.max():.2f}")
        else:
            print("  never ahead of both reference strategies")
        print(band.round(4).to_string(index=False))
        print()

### Feature importance

In [ ]:
if exploratory_importance:
    for model, (method, frame) in sorted(exploratory_importance.items()):
        print(f"{model}, top {TOP_N} by {method} importance "
              f"({IMPORTANCE_CONDITION})")
        print(frame.round(4).to_string(index=False))
        trend = frame["feature"].str.endswith("_trend").sum()
        print(f"  trend features in the top {TOP_N}: {trend}/{TOP_N}")
        print()
else:
    print("no importance file held back for the exploratory model")

## 6. Save outputs

The head-to-head comparison table is written to `data/` as CSV and Markdown,
with a second CSV covering every model including temporal_ffnn. The summary
chart and the additional graphs are written to `figures/` as PNG. The
calibration, lead-time and feature-importance tables are each written as CSV
only if they could be built, with the calibration table also as Markdown.

In [ ]:
def to_markdown_table(frame):
    """Render a DataFrame as a pipe-delimited Markdown table."""
    columns = list(frame.columns)
    lines = ["| " + " | ".join(columns) + " |",
             "|" + "|".join("---" for _ in columns) + "|"]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |")
    return "\n".join(lines) + "\n"


csv_path = DATA_DIR / "model_comparison_table.csv"
md_path = DATA_DIR / "model_comparison_table.md"
chart_path = FIGURES_DIR / "model_comparison_chart.png"
grid_path = FIGURES_DIR / "model_comparison_metric_grid.png"
spec_path = FIGURES_DIR / "model_comparison_specificity.png"
pair_path = FIGURES_DIR / "model_comparison_auprc_vs_lift.png"

FIGURES_DIR.mkdir(exist_ok=True)

comparison.round(4).to_csv(csv_path, index=False)
md_path.write_text(to_markdown_table(comparison_display))
comparison_all.round(4).to_csv(DATA_DIR / "model_comparison_table_all_models.csv",
                               index=False)
fig.savefig(chart_path, dpi=200, bbox_inches="tight")
fig_grid.savefig(grid_path, dpi=200, bbox_inches="tight")
fig_spec.savefig(spec_path, dpi=200, bbox_inches="tight")
fig_pair.savefig(pair_path, dpi=200, bbox_inches="tight")

written = [csv_path, md_path, chart_path, grid_path, spec_path, pair_path]

if calibration_table is not None:
    cal_csv = DATA_DIR / "model_comparison_calibration.csv"
    cal_md = DATA_DIR / "model_comparison_calibration.md"
    calibration_table.round(4).to_csv(cal_csv, index=False)
    cal_md.write_text(to_markdown_table(calibration_table.round(4)))
    written += [cal_csv, cal_md]
else:
    print("calibration table not written: no calibration numbers in the result files")

if lead_time_table is not None:
    lead_csv = DATA_DIR / "model_comparison_lead_time.csv"
    lead_time_table.round(2).to_csv(lead_csv, index=False)
    written.append(lead_csv)

if importance_by_model:
    imp_csv = DATA_DIR / "model_comparison_feature_importance.csv"
    combined.to_csv(imp_csv, index=False)
    written.append(imp_csv)

print()
for p in written:
    print(f"{p.name:44s} {p.stat().st_size / 1024:7.1f} KB")

## Summary

Final counts, and a plain statement of which comparisons are complete and which
are still pending.

In [ ]:
print(f"models compared: {comparison['model'].nunique()}")
print(f"model/condition combinations: {len(comparison)}")
print()
print("best lift per model:")
for model, group in comparison.groupby("model"):
    best = group.loc[group["lift"].idxmax()]
    print(f"  {model:16s} {best['lift']:.3f}  ({best['condition']}, "
          f"AUPRC {best['AUPRC']:.4f} at prevalence {best['prevalence'] * 100:.2f}%)")
print()

print("data completeness:")
print(f"  confidence intervals: "
      f"{'complete' if not missing_ci else 'missing for ' + str(missing_ci)}")
print(f"  calibration numbers:  "
      f"{'complete' if calibration_table is not None and not missing_calibration else 'incomplete'}")
print(f"  feature importance:   "
      f"{len(importance_by_model)}/{comparison['model'].nunique()} models")
if unrecognised:
    print(f"  files excluded for an unrecognised condition name: {len(unrecognised)}")